In [0]:
%sql
USE CATALOG students_data;
USE SCHEMA `team3-taxi`;

In [0]:
%sql
CREATE OR REPLACE TABLE dim_vendor AS
SELECT
    ROW_NUMBER() OVER (ORDER BY vendor_id) AS vendor_key,
    vendor_id,
    CASE vendor_id
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'VeriFone Inc.'
        ELSE 'Unknown'
    END AS vendor_name
FROM (
    SELECT DISTINCT vendor_id
    FROM fact_trip_silver
);


In [0]:
%sql
CREATE OR REPLACE TABLE dim_rate_code AS
SELECT
    ROW_NUMBER() OVER (ORDER BY rate_code_id) AS rate_code_key,
    rate_code_id,
    CASE rate_code_id
        WHEN 1 THEN 'Standard rate'
        WHEN 2 THEN 'JFK'
        WHEN 3 THEN 'Newark'
        WHEN 4 THEN 'Nassau or Westchester'
        WHEN 5 THEN 'Negotiated fare'
        WHEN 6 THEN 'Group ride'
        ELSE 'Unknown'
    END AS rate_code_desc
FROM (
    SELECT DISTINCT rate_code_id
    FROM fact_trip_silver
);


In [0]:
%sql
CREATE OR REPLACE TABLE dim_payment_type AS
SELECT
    ROW_NUMBER() OVER (ORDER BY payment_type_id) AS payment_type_key,
    payment_type_id,
    CASE payment_type_id
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
        ELSE 'Other'
    END AS payment_type_desc
FROM (
    SELECT DISTINCT payment_type_id
    FROM fact_trip_silver
);


In [0]:
%sql
CREATE OR REPLACE TABLE dim_location AS
SELECT
    ROW_NUMBER() OVER (ORDER BY latitude, longitude) AS location_key,
    latitude,
    longitude
FROM (
    SELECT DISTINCT pickup_latitude AS latitude, pickup_longitude AS longitude
    FROM fact_trip_silver
    UNION
    SELECT DISTINCT dropoff_latitude AS latitude, dropoff_longitude AS longitude
    FROM fact_trip_silver
);


In [0]:
%sql
    
CREATE OR REPLACE TABLE dim_datetime_day AS
WITH dates AS (
    SELECT DATEADD(day, seq_num, '2015-01-01') AS date_value
    FROM (SELECT EXPLODE(SEQUENCE(0, 30)) AS seq_num)  -- Jan 2015
    UNION ALL
    SELECT DATEADD(day, seq_num, '2016-01-01') AS date_value
    FROM (SELECT EXPLODE(SEQUENCE(0, 89)) AS seq_num)  -- Jan–Mar 2016
)
SELECT
    ROW_NUMBER() OVER (ORDER BY date_value) AS datetime_day_key,
    date_value,
    YEAR(date_value) AS year,
    MONTH(date_value) AS month,
    DAY(date_value) AS day,
    DAYOFWEEK(date_value) AS day_of_week,
    CASE WHEN DAYOFWEEK(date_value) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend
FROM dates
ORDER BY date_value;

In [0]:
%sql
    
CREATE OR REPLACE TABLE dim_datetime_hour AS
WITH hours_2015 AS (
    SELECT DATEADD(hour, seq_num, '2015-01-01 00:00:00') AS datetime_value
    FROM (SELECT EXPLODE(SEQUENCE(0, 31 * 24 - 1)) AS seq_num)
),
hours_2016 AS (
    SELECT DATEADD(hour, seq_num, '2016-01-01 00:00:00') AS datetime_value
    FROM (SELECT EXPLODE(SEQUENCE(0, 90 * 24 - 1)) AS seq_num)
),
all_hours AS (
    SELECT * FROM hours_2015
    UNION ALL
    SELECT * FROM hours_2016
)
SELECT
    ROW_NUMBER() OVER (ORDER BY datetime_value) AS datetime_hour_key,
    datetime_value,
    TO_DATE(datetime_value) AS date_value,
    YEAR(datetime_value) AS year,
    MONTH(datetime_value) AS month,
    DAY(datetime_value) AS day,
    HOUR(datetime_value) AS hour,
    DAYOFWEEK(datetime_value) AS day_of_week,
    CASE WHEN DAYOFWEEK(datetime_value) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend
FROM all_hours
ORDER BY datetime_value;

In [0]:
%sql
    
CREATE OR REPLACE TABLE dim_datetime_minute AS
WITH minutes_2015 AS (
    SELECT DATEADD(minute, seq_num, '2015-01-01 00:00:00') AS datetime_value
    FROM (SELECT EXPLODE(SEQUENCE(0, 31 * 24 * 60 - 1)) AS seq_num)
),
minutes_2016 AS (
    SELECT DATEADD(minute, seq_num, '2016-01-01 00:00:00') AS datetime_value
    FROM (SELECT EXPLODE(SEQUENCE(0, 90 * 24 * 60 - 1)) AS seq_num)
),
all_minutes AS (
    SELECT * FROM minutes_2015
    UNION ALL
    SELECT * FROM minutes_2016
)
SELECT
    ROW_NUMBER() OVER (ORDER BY datetime_value) AS datetime_minute_key,
    datetime_value,
    TO_DATE(datetime_value) AS date_value,
    YEAR(datetime_value) AS year,
    MONTH(datetime_value) AS month,
    DAY(datetime_value) AS day,
    HOUR(datetime_value) AS hour,
    MINUTE(datetime_value) AS minute,
    DAYOFWEEK(datetime_value) AS day_of_week,
    CASE WHEN DAYOFWEEK(datetime_value) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend
FROM all_minutes
ORDER BY datetime_value;